# Spam Detection - Baseline Notebook
**Öğrenci 1 | TF-IDF + Logistic Regression Baseline + PyTorch DataLoader**

## 1. Data Loading + İlk Bakış

In [ ]:
import pandas as pd

file_path = "../data/raw/spam.csv"

df = pd.read_csv(file_path, encoding='latin-1')

print(df.head())
print("\nColumns:", df.columns.tolist())
print("\nShape:", df.shape)

## 2. Dataset Düzenleme

In [ ]:
# Gereksiz sütunları at, sadece v1 ve v2'yi al
df = df[['v1', 'v2']]

# Kolon isimlerini değiştir
df.columns = ['label', 'text']

df.head()

## 3. Temizlik

In [ ]:
# Boş değerleri sil
df = df.dropna()

# Duplicate kontrolü
df = df.drop_duplicates()

print("After cleaning:", df.shape)

## 4. Label Encoding

In [ ]:
# ham -> 0, spam -> 1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# Dağılım
print(df['label'].value_counts())
print("\nClass distribution (%):\n", df['label'].value_counts(normalize=True).round(3) * 100)

## 5. Train / Val / Test Split (Stratified)

In [ ]:
from sklearn.model_selection import train_test_split

X = df['text']
y = df['label']

# %70 train, %30 temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# %15 val, %15 test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train:", len(X_train))
print("Val:  ", len(X_val))
print("Test: ", len(X_test))

## 6. TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)
X_test_tfidf  = vectorizer.transform(X_test)

print("TF-IDF matrix shape (train):", X_train_tfidf.shape)

## 7. Baseline Model — Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

print("Accuracy: ", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:   ", round(recall_score(y_test, y_pred), 4))
print("F1 Score: ", round(f1_score(y_test, y_pred), 4))
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

## 8. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

print(cm)
# Beklenen yapı:
# [[ham doğru,   ham→spam hata ]
#  [spam→ham hata, spam doğru  ]]

## 8. PyTorch Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Sparse → dense → tensor
X_train_tensor = torch.tensor(X_train_tfidf.toarray(), dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val_tfidf.toarray(),   dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_tfidf.toarray(),  dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_val_tensor   = torch.tensor(y_val.values,   dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test.values,  dtype=torch.float32)

# Dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor,   y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor,  y_test_tensor)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)

print("DataLoader hazır")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

# Sanity check
X_batch, y_batch = next(iter(train_loader))
print(f"Batch shape: X={X_batch.shape}, y={y_batch.shape}")